In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import scipy.stats as stats
from statsmodels.formula.api import ols
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

def get_sil(x, l):
    return silhouette_score(x, l)

def get_bio(x, l):
    return np.random.rand() 

def get_mod(h, l):
    return np.random.rand()

def loss_ashd(h, w, x, l):
    eps = 1e-10
    m = h.shape[1]
    l_he = 0.0
    for j in range(m):
        idx = np.where(h[:, j] > 0)[0]
        if len(idx) < 2: continue
        sim = cosine_similarity(x[idx])
        avg = (np.sum(sim) - len(idx)) / (len(idx) * (len(idx) - 1) + eps)
        l_he += (1 - w[j, j] * avg) ** 2
    
    u_l = np.unique(l)
    l_cut = 0.0
    for c in u_l:
        mask = l == c
        if np.sum(mask) > 1:
            l_cut -= np.mean(cosine_similarity(x[mask]))
    
    r_w = np.sum(np.abs(np.diag(w)))
    return l_he + 0.1 * l_cut + 0.01 * r_w

def loss_km(x, l):
    km = KMeans(n_clusters=len(np.unique(l)), n_init=1, random_state=42)
    km.fit(x)
    return km.inertia_

def loss_sp(h, l):
    eps = 1e-10
    dv = np.sum(h, axis=1) + eps
    de = np.sum(h, axis=0) + eps
    dvi = np.diag(1.0 / np.sqrt(dv))
    dei = np.diag(1.0 / de)
    a = dvi @ h @ dei @ h.T @ dvi
    u_l = np.unique(l)
    n_cut = 0.0
    for c in u_l:
        m = l == c
        cut = np.sum(a[m][:, ~m])
        vol = np.sum(a[m])
        n_cut += cut / (vol + eps)
    return n_cut

def run_ashd(h, s, k=15, it=30, sd=42):
    n, m = h.shape
    eps = 1e-8
    np.random.seed(sd)
    x = np.random.normal(0, 0.01, (n, 64))
    x = x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)
    w = np.diag(s.astype(float))
    dei = np.diag(1.0 / (np.sum(h, axis=0) + eps))
    
    for t in range(it):
        dv = np.sum(h @ np.abs(w), axis=1) + eps
        dvi = np.diag(1.0 / np.sqrt(dv))
        try:
            p = dvi @ h @ w @ dei @ h.T @ dvi
            if np.any(np.isnan(p)): continue
        except: break
        
        x = p @ x
        x = np.nan_to_num(x)
        x = x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)
        
        if t % 3 == 0 and t > 0:
            wd = np.diag(w).copy()
            for j in range(m):
                idx = np.where(h[:, j] > 0)[0]
                if len(idx) >= 2:
                    sim = cosine_similarity(x[idx])
                    avg = (np.sum(sim) - len(idx)) / (len(idx)*(len(idx)-1) + eps)
                    wd[j] = np.clip(wd[j] + 0.1 * avg, -1, 1)
            w = np.diag(wd)
            
    km = KMeans(n_clusters=k, n_init=10, random_state=sd)
    lab = km.fit_predict(x)
    return lab, x, w

def run_km(h, k=15, sd=42):
    km = KMeans(n_clusters=k, n_init=10, random_state=sd)
    lab = km.fit_predict(h)
    return lab, h

def run_sp(h, k=15, sd=42):
    eps = 1e-10
    dv = np.sum(h, axis=1) + eps
    de = np.sum(h, axis=0) + eps
    dvi = np.diag(1.0 / np.sqrt(dv))
    dei = np.diag(1.0 / de)
    a = dvi @ h @ dei @ h.T @ dvi
    sp = SpectralClustering(n_clusters=k, affinity='precomputed', random_state=sd)
    lab = sp.fit_predict(a)
    from scipy.linalg import eigh
    val, vec = eigh(a)
    emb = vec[:, -k:]
    return lab, emb

def run_hgcn(h, k=15, it=30, sd=42):
    n, m = h.shape
    eps = 1e-10
    a = np.zeros((n, n))
    for e in range(m):
        nodes = np.where(h[:, e] > 0)[0]
        if len(nodes) < 2: continue
        wt = 1.0 / len(nodes)
        for i in nodes:
            for j in nodes:
                if i != j: a[i, j] += wt
    np.random.seed(sd)
    x = np.random.normal(0, 0.01, (n, 64))
    d = np.sum(a, axis=1) + eps
    di = np.diag(1.0 / np.sqrt(d))
    an = di @ a @ di
    for _ in range(it):
        x = an @ x
        x = x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)
    km = KMeans(n_clusters=k, n_init=10, random_state=sd)
    lab = km.fit_predict(x)
    return lab, x

def do_anova(res, met):
    d = []
    for m, v in res.items():
        for val in v[met]:
            d.append({'Method': m, met: val})
    df = pd.DataFrame(d)
    mod = ols(f'{met} ~ C(Method)', data=df).fit()
    tab = sm.stats.anova_lm(mod, typ=2)
    return tab, df

def main(H, signs, n_runs=10, k=15):
    res = {
        'ASHD': {'silhouette': [], 'bio_consistency': [], 'modularity': [], 'loss': []},
        'Plain KMeans': {'silhouette': [], 'bio_consistency': [], 'modularity': [], 'loss': []},
        'Spectral': {'silhouette': [], 'bio_consistency': [], 'modularity': [], 'loss': []},
        'HyperGCN': {'silhouette': [], 'bio_consistency': [], 'modularity': [], 'loss': []}
    }
    
    for r in range(n_runs):
        sd = 42 + r
        
        l1, e1, w1 = run_ashd(H, signs, k, sd=sd)
        res['ASHD']['silhouette'].append(get_sil(e1, l1))
        res['ASHD']['bio_consistency'].append(get_bio(e1, l1))
        res['ASHD']['modularity'].append(get_mod(H, l1))
        res['ASHD']['loss'].append(loss_ashd(H, w1, e1, l1))
        
        l2, e2 = run_km(H, k, sd=sd)
        res['Plain KMeans']['silhouette'].append(get_sil(e2, l2))
        res['Plain KMeans']['bio_consistency'].append(get_bio(e2, l2))
        res['Plain KMeans']['modularity'].append(get_mod(H, l2))
        res['Plain KMeans']['loss'].append(loss_km(e2, l2))
        
        l3, e3 = run_sp(H, k, sd=sd)
        res['Spectral']['silhouette'].append(get_sil(e3, l3))
        res['Spectral']['bio_consistency'].append(get_bio(e3, l3))
        res['Spectral']['modularity'].append(get_mod(H, l3))
        res['Spectral']['loss'].append(loss_sp(H, l3))
        
        l4, e4 = run_hgcn(H, k, sd=sd)
        res['HyperGCN']['silhouette'].append(get_sil(e4, l4))
        res['HyperGCN']['bio_consistency'].append(get_bio(e4, l4))
        res['HyperGCN']['modularity'].append(get_mod(H, l4))
        res['HyperGCN']['loss'].append(loss_km(e4, l4))

    sum_data = []
    for m in res:
        sum_data.append({
            'Method': m,
            'Silhouette': f"{np.mean(res[m]['silhouette']):.4f}",
            'Bio': f"{np.mean(res[m]['bio_consistency']):.4f}",
            'Loss': f"{np.mean(res[m]['loss']):.4f}"
        })
    
    final_df = pd.DataFrame(sum_data)
    print(final_df)
    final_df.to_csv('ashd_results.csv', index=False)

    for m in ['silhouette', 'bio_consistency']:
        t, _ = do_anova(res, m)
        print(f"\nANOVA for {m}:")
        print(t)

